## 실행 환경 확인

이 셀은 Jupyter Notebook과 Python 실행 환경이 정상적으로 동작하는지 확인하기 위한 예시 코드임.

- Python이 정상적으로 실행되는지 점검
- 노트북 셀 실행이 되는지 확인
- 이후 분석 코드가 문제없이 돌기 위한 기본 체크

In [1]:
print("hello world")

hello world


## 데이터 샘플 확인

이 코드는 실제 GA4 이벤트 데이터의 형태를 살펴보는 단계임.

- 테이블에서 일부 행을 불러와 데이터가 제대로 들어왔는지 확인



In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="pro-talon-503713-s3")

query = """
SELECT *
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210131`
LIMIT 10
"""

df = client.query(query).to_dataframe()
print(df.head())
print(df.columns.tolist())

  event_date   event_timestamp       event_name  \
0   20210131  1612069510766593        page_view   
1   20210131  1612069529243877           scroll   
2   20210131  1612069515781635        page_view   
3   20210131  1612069530073506  user_engagement   
4   20210131  1612069510766593    session_start   

                                        event_params  \
0  [{'key': 'gclid', 'value': {'string_value': No...   
1  [{'key': 'debug_mode', 'value': {'string_value...   
2  [{'key': 'debug_mode', 'value': {'string_value...   
3  [{'key': 'page_location', 'value': {'string_va...   
4  [{'key': 'ga_session_number', 'value': {'strin...   

   event_previous_timestamp  event_value_in_usd  event_bundle_sequence_id  \
0                      <NA>                 NaN                6595101026   
1                      <NA>                 NaN                9011338476   
2                      <NA>                 NaN               -6830522854   
3                      <NA>                 NaN 

##  이벤트별 집계 확인

이 코드는 어떤 이벤트가 얼마나 많이 발생했는지 확인하는 단계임.

- `event_name` 기준으로 이벤트 수 집계
- 각 이벤트의 총 발생 건수 확인
- 사용자 수(`unique_users`)도 함께 보며 이벤트의 실제 참여 규모를 파악

주요 확인 포인트:
- 어떤 이벤트가 발생하는지
- 사용자 기반으로도 규모가 큰 이벤트가 있는지
- acquisition 분석에서 핵심 이벤트를 선별할 수 있는지

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project="pro-talon-503713-s3")

query = """
SELECT
    event_name,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY event_name
ORDER BY event_count DESC
"""

df = client.query(query).to_dataframe()
print(df)

             event_name  event_count  unique_users
0             page_view      1350428        269792
1       user_engagement      1058721        213004
2                scroll       493072        138098
3             view_item       386068         61252
4         session_start       354970        267116
5           first_visit       257462        257314
6        view_promotion       190104        102443
7           add_to_cart        58543         12545
8        begin_checkout        38757          9715
9           select_item        31007         13180
10  view_search_results        26172         14449
11    add_shipping_info        19722          9714
12     add_payment_info        13899          5751
13     select_promotion         9450          8164
14             purchase         5692          4419
15                click         1446          1010
16       view_item_list           71            44


##  이벤트 파라미터별 확인

이 코드는 각 이벤트에서 어떤 매개변수들이 얼마나 많이 발생하는지 확인하는 단계임.

- `param.key` 기준으로 어떤 파라미터가 자주 나오는지 집계
- 각 이벤트별로 어떤 정보가 수집되고 있는지 확인

주요 확인 포인트:
- 사용자 유입 경로, 캠페인, 소스 등에 쓰이는 파라미터 키를 찾는지
- acquisition 관련 파라미터가 어떤 이름으로 저장되는지

In [3]:
from google.cloud import bigquery

client = bigquery.Client(project="pro-talon-503713-s3")

query = """
SELECT
    event_name,
    param.key AS param_key,
    COUNT(*) AS occurrence_count
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`,
UNNEST(event_params) AS param
WHERE _TABLE_SUFFIX BETWEEN '20201101' AND '20210131'
GROUP BY event_name, param_key
ORDER BY event_name, occurrence_count DESC
"""

df = client.query(query).to_dataframe()
print(df)

              event_name          param_key  occurrence_count
0       add_payment_info  ga_session_number             13899
1       add_payment_info    session_engaged             13899
2       add_payment_info         page_title             13899
3       add_payment_info           currency             13899
4       add_payment_info      page_location             13899
..                   ...                ...               ...
275  view_search_results        clean_event             17771
276  view_search_results             source               361
277  view_search_results           campaign               361
278  view_search_results             medium               361
279  view_search_results               term                31

[280 rows x 3 columns]


##  first_visit 관련 핵심 파라미터 추출

이 코드는 `first_visit` 이벤트에서 실제로 어떤 파라미터가 중요한지 확인하는 마지막 단계임.

- `first_visit` 이벤트만 필터링
- `occurrence_count` 기준으로 내림차순 정렬
- acquisition 분석에서 가장 자주 확인해야 할 파라미터를 추출

주요 확인 포인트:
- `first_visit` 이벤트에서 어떤 파라미터가 자주 발생하는지
- 유입 경로, 캠페인, 채널 관련 키를 찾는지
- 이후 실제 acquisition 분석의 기준 변수를 정립하는지

In [4]:
first_visit_params = df[df['event_name'] == 'first_visit'].sort_values('occurrence_count', ascending=False)
print(first_visit_params)

     event_name              param_key  occurrence_count
84  first_visit      ga_session_number            257462
85  first_visit          page_location            257462
86  first_visit          ga_session_id            257462
87  first_visit  engaged_session_event            257368
88  first_visit        session_engaged            257178
89  first_visit             page_title            254591
90  first_visit          page_referrer            165332


## 추가 분석 포인트: 상위 컬럼으로 분류되는 유입 변수

신규 유입을 세분화해서 분석할 때, 지역, 채널과 같은 변수는 이벤트 파라미터 안에 있는 값으로 확인되는 것이 아니라, 별도의 상위 컬럼으로 저장돼 있음.

예를 들어:
- 지역 정보는 `geo` 컬럼으로 존재함.
- 채널 정보는 `traffic_source` 컬럼으로 존재함.

즉, `event_params`에서 파라미터를 확인하는 것만으로는 신규 유입의 세부 분류를 이해하기 어렵고, 상위 컬럼을 살펴봐야 함.